# Lesson 5.2: Search, filter and return the evidence

If a withdrawn policy contains the best search terms, Harbour still should not use it. First identify which records the rules allow, then see how word matches rank those records. Follow the source identifier into the text the assistant would receive so the evidence can be checked later.

[Companion lesson](../modules/05-rag/02-code-workshop.md) · [Original-source walkthrough](../modules/05-rag/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Identify which source is allowed into the search result. Then read the assembled context and find the source identifier attached to the text.

In [ ]:
records = [
    {"id": "P-1", "scope": "service", "current": True,
     "text": "Finance reviews disputed delivery credits"},
    {"id": "P-0", "scope": "service", "current": False,
     "text": "Account managers approve delivery credits"},
    {"id": "H-1", "scope": "restricted", "current": True,
     "text": "Staff credit eligibility depends on employment terms"},
]

def retrieve(question, records, permitted_scopes, k=2):
    terms = set(question.lower().split())
    candidates = []
    for record in records:
        if record["scope"] not in permitted_scopes or not record["current"]:
            continue
        words = set(record["text"].lower().split())
        score = len(terms & words)
        if score > 0:
            candidates.append((score, record))
    candidates.sort(key=lambda pair: pair[0], reverse=True)
    return [record for score, record in candidates[:k]]

evidence = retrieve("Who reviews delivery credits", records, {"service"})

def make_context(evidence):
    if not evidence:
        return "No permitted current evidence was found."
    return "\n\n".join(f"[{r['id']}] {r['text']}" for r in evidence)

context = make_context(evidence)

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
print([r["id"] for r in evidence])
print(context)
assert [r["id"] for r in evidence] == ["P-1"]
assert all(f"[{r['id']}]" in context for r in evidence)

## Try a change

Search using an empty permission set. What should the text supplied to the assistant say?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

The context says that no permitted current evidence was found. It does not substitute a withdrawn or restricted source.

```python
none_allowed = retrieve("Who reviews delivery credits", records, set())
assert none_allowed == []
print(make_context(none_allowed))
```

</details>